<a href="https://colab.research.google.com/github/Zyan-W/Auto-MFA/blob/main/auto_mfa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ASR-assisted MFA pipeline for phoentic annotation: a practice


---
20260517

by Wang Zhiyan

---

## What is this file?
This file is a Jupyter Python notebook, which looks like a markdown file and can allow you run Python code directly in this file.


## What is file for?
I've always been dreaming a method that can automatically do the annotating, aligning, and segmenting for me.

Now, with the help of Automatic Speech Recognition(ASR) model and Montreal Forced Aligner(MFA), the fully-automatic pipeline becomes possible.

Here, this file provides a basic solution of automatically extracting the audio transcription with time stamp using ASR model by OpenAI named "Whisper". The sentence-level transcription can be used in the automatic annotation and alignment.


## How does it work?
Whisper model is a robust model trained by OpenAI.

It can recognize different languages and speech with accents.

More details can be found here.

[Whisper on Github](https://github.com/openai/whisper)


## How should I use it?
Whisper model can be demanding when it's run on a old PC, so you can run this script on a Google Colab server, which is basically free and should be powerful enough.

You can use it in Google colab (recommend), and run code with connection to GPUs on Google server for FREE.



## Run in Colab
You can save a copy to YOUR own google drive, and run it in Colab.

Click the button on the up-right corner to change the runtime to "T4 GPU", which is a powerful processor by NVidia.

Run the code below to check if the GPU on the server is ready to go.

In [ ]:
!nvidia-smi


If it's ready, you should see the "Tesla T4", the power usage, and memory usage etc,.

Now let's play.

## Step 1: Prepare the model

install Whisper & FFmpeg

In [ ]:
!pip install -U openai-whisper
!apt-get update -qq
!apt-get install -y ffmpeg

## Step 2: Transcribe the audio

There are two methods from which you can choose to load your audio files.

### Method 1: Direct audio upload
create a new folder on the server for audio upload

In [ ]:
!cd /content/
!mkdir -p raw_audio/
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
  !mv "{filename}" /content/raw_audio/


In [ ]:
!cd /content/
!mkdir -p output/
for filename in uploaded.keys():
  print(filename)
for filename in uploaded.keys():
  !whisper "{filename}" --model small --language ja --output_dir output --output_format all

### Method 2: Access audio files in Google drive
Instead of upload audio file to server, you can also access the audio files in Google drive directly.

BECAUSE OF THE PERMISSIOIN,  ACTIONS WITHOUT CAUTION MAY CAUSE DATA CORRUPTION OR DELETION.

CONNECT, USE, CODE AT YOUR OWN RISK.

ESPECIALLY, YOU SHOULD NOT RUN CODE LIKE `# rm -rf`
IN YOUR GOOGLE DRIVE FOLDER.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

If Google drive is mounted successfully, you should see `"Mounted at /content/drive
"`.

Then, take the folder `"/auto-mfa/"` in YOUR Google drive as an example.

You should upload all the audio files (e.g., *.wav files) to a folder named `"raw_audio"` under `"auto-mfa"`.

In [ ]:
%cd /content/drive/MyDrive/auto-mfa
!mkdir -p output
!whisper raw_audio/*.wav --model small --language ja --output_dir output --output_format all


## Step 3: Turn the transcription into TextGrid

To save the necessary content from transcription files such as .json files, we need to use the function written below.

In [ ]:
!pip install praatio # a python module designed for Praat

In [ ]:
from pathlib import Path
import json

from praatio import textgrid
from praatio.utilities.constants import Interval


def whisper_json_to_textgrid(
    json_path: str | Path,
    tg_path: str | Path,
    tier_name: str = "sentences",
) -> None:
    """
    Convert one Whisper JSON file to one Praat TextGrid file.
    """

    json_path = Path(json_path)
    tg_path = Path(tg_path)

    # 自动补上 .TextGrid 后缀
    if tg_path.suffix != ".TextGrid":
        tg_path = tg_path.with_suffix(".TextGrid")

    # 确保输出文件夹存在
    tg_path.parent.mkdir(parents=True, exist_ok=True)

    # 读取 Whisper JSON
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    entries = []

    # 提取并清理 Whisper segments
    for seg in data["segments"]:
        start = float(seg["start"])
        end = float(seg["end"])
        label = str(seg["text"]).strip()

        if end > start and label:
            entries.append(Interval(start, end, label))

    if not entries:
        raise ValueError(f"No valid segments found in {json_path}")

    min_time = 0.0
    max_time = max(entry.end for entry in entries)

    tg = textgrid.Textgrid()

    tier = textgrid.IntervalTier(
        tier_name,
        entries,
        minT=min_time,
        maxT=max_time,
    )

    tg.addTier(tier)

    tg.save(
        str(tg_path),
        format="long_textgrid",
        includeBlankSpaces=True,
    )

    print(f"Saved: {tg_path}")


def batch_convert_whisper_jsons(
    input_dir: str | Path,
    output_dir: str | Path,
    tier_name: str = "sentences",
) -> None:
    """
    Convert all .json files in input_dir to TextGrid files in output_dir.
    """

    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    if not input_dir.exists():
        raise FileNotFoundError(f"Input directory does not exist: {input_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)

    json_files = sorted(input_dir.glob("*.json"))

    if not json_files:
        raise FileNotFoundError(f"No JSON files found in: {input_dir}")

    print(f"Found {len(json_files)} JSON file(s).")

    for json_path in json_files:
        tg_path = output_dir / json_path.stem

        whisper_json_to_textgrid(
            json_path=json_path,
            tg_path=tg_path,
            tier_name=tier_name,
        )



In [ ]:

# ===== change the dir to your directory =====

batch_convert_whisper_jsons(
    input_dir="/content/drive/MyDrive/auto-mfa/output",
    output_dir="/content/drive/MyDrive/auto-mfa/mfa-input",
    tier_name="sentences",
)

## Step 4: MFA

In [ ]:
!cp -rv /content/drive/MyDrive/auto-mfa/raw_audio/* /content/drive/MyDrive/auto-mfa/mfa-input
!ls /content/drive/MyDrive/auto-mfa/mfa-input

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()
!which mamba
!conda --version

In [ ]:
!mamba create -n aligner -c conda-forge python=3.11 montreal-forced-aligner -y
!mamba run -n aligner mfa model download acoustic japanese_mfa
!mamba run -n aligner mfa model download dictionary japanese_mfa
!mamba install -n aligner -c conda-forge spacy sudachipy sudachidict-core

In [ ]:
!mamba run -n aligner mfa align /content/drive/MyDrive/auto-mfa/mfa-input/ japanese_mfa japanese_mfa /content/drive/MyDrive/auto-mfa/mfa-output/ --clean

## Done!

Now you can download the .TextGrid files and check if they work for you!